# Synthetic biomarker benchmark

The generated pages hold the facts: [how it is run](../docs/benchmarks/biomarker-synthetic-docs.md)
and [what came out](../docs/benchmarks/biomarker-synthetic-results.md). The shapes themselves, and
the values their geometry requires, are in [the shapes
notebook](biomarker-synthetic-shapes.ipynb). This notebook is where the judgement goes.

**Everything here is measured over synthetically generated segmentations** — arteries, veins and an
optic disc *drawn* from equations, at a stated scale, on a blank field. No eye was photographed and
no annotator was consulted. That is the whole point: when the shape is drawn rather than traced,
what it ought to measure is not a matter of opinion.

## Two kinds of benchmarking, and they are different questions

1. **Sensitivity to rotation.** Every shape is drawn again at several angles, in continuous
   coordinates, rather than by turning a picture. The geometry is *identical* at each angle, so
   anything that moves is the implementation or the pixel grid beneath it. This needs no ground
   truth at all — it is a self-consistency check, and a program can fail it badly while still
   landing on the right answer on average.
2. **Agreement with ground truth.** The ground truth here is a **theoretical value stored alongside
   the images**, derived from the geometry before a single pixel was drawn. It is not an
   annotation and it is not another program's output. An implementation that disagrees with it is
   *wrong*, not merely different.

The two can disagree about a program, and when they do that is itself the finding: a steady wrong
number and a jittery right one are different diseases with different cures.

## How to read the tables

Unless a section says otherwise, every table has **one row per canonical biomarker name** and **one
column per implementation** — the opposite way round from every other benchmark here, where the
model leads. It is that way because the question is *do these programs compute the same quantity*,
and a reader answering it reads **across a row**. Where a quantity was measured on several shapes
or at several angles, the cell holds the **worst** of them: a mean would hide exactly the case
worth knowing about.

**The columns stand in lineage order, not alphabetically**, and the order carries information:

| Columns | Why they stand together |
| --- | --- |
| `pvbm`, `ocular` | OCULAR imports PVBM's tortuosity, perimeter and branching-angle helpers at runtime and measures with a modified copy of its `CREVBMs` |
| `automorph`, `automorphalyzer`, `automorphclass` | both descendants are rewrites of AutoMorph's measuring stage |
| `vascx` | shares no code with any of the above; it reimplements every biomarker from the definitions |

So a difference **between neighbours** is a change somebody made on purpose, and a difference
**across a boundary** is two programs that never shared a line of code arriving at different
numbers for the same definition.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def repository() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(f"nothing above {Path.cwd()} looks like the fundus-atlas repository")


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))

from benchmarks import biomarker_synthetic as benchmark  # noqa: E402
from benchmarks.shapes import store  # noqa: E402
from biomarkers.utils import catalogue  # noqa: E402

#: The two thresholds this notebook reports against. **Neither is a standard anybody agreed**: they
#: are reporting conveniences, chosen so that a table shows the handful of quantities worth looking
#: at rather than two hundred rows of nothing. Nothing passes or fails a benchmark here.
ROTATION_TOLERANCE = 0.10
GROUND_TRUTH_TOLERANCE = 0.25

#: The noise floor, in **biomarker units**, and it does two jobs: a difference smaller than this is
#: treated as agreement, and no percentage is ever taken against a denominator smaller than this.
#:
#: **It is deliberately, unusually high.** A tolerance of 0.01 would be absurd for a physical
#: constant; here it is a way of not reporting the rasteriser. These shapes are drawn onto a pixel
#: lattice and skeletonised, and that quantisation puts a floor under how exactly *any* program can
#: answer — below which a percentage is a ratio of two noises and says nothing about the software.
#:
#: **It is not free, and what it costs is worth knowing before reading any table below.** Five
#: catalogued biomarkers have typical values *smaller* than this floor, so for them the floor is
#: doing most of the arithmetic rather than a little of it:
#:
#: | Biomarker | Typical value the geometry requires |
#: | --- | --- |
#: | `tortuosity/grisan-density/{artery,vein}` | 0.0011 |
#: | `tortuosity/spline-mean-curvature/{artery,vein}` | 0.0053 |
#: | `vascular-density/over-image/vessels` | 0.0081 |
#:
#: For those five, an error shown below is measured against 0.01 rather than against the true
#: value, so it is **understated** — by up to tenfold for Grisan density — and a wrong answer
#: smaller than 0.01 is forgiven outright. Everything else in the catalogue is one or larger and is
#: barely touched. Raising this constant makes the tables kinder in exactly one direction, and the
#: honest reading of a small error on one of those five is "under the noise floor", not "correct".
EPS = 0.01

#: Lineage order, from the benchmark itself, so the columns of every table below stand the way the
#: intro describes. Never re-sort this.
IMPLEMENTATIONS = list(benchmark.IMPLEMENTATIONS)
EVIDENCE = ROOT / "results" / benchmark.NAME
STORE = ROOT / store.STORE
pd.set_option("display.width", 200, "display.max_rows", 300, "display.max_columns", 40)

In [2]:
def measured(slug: str) -> pd.DataFrame:
    """Everything one implementation returned, over every shape and every angle."""
    frames = [pd.read_csv(path) for path in sorted((EVIDENCE / slug).glob("*.csv"))]
    frame = pd.concat(frames, ignore_index=True)
    frame.insert(0, "implementation", slug)
    return frame


#: What each program said, under **its own** column names, exactly as the run wrote it.
MEASURED = {slug: measured(slug) for slug in IMPLEMENTATIONS}

#: Which catalogued biomarker each of those columns is believed to answer to, or `None` where the
#: catalogue has no name for that quantity. This is the adapter's claim, not the run's: a run
#: records no mapping at all, which is what lets the same evidence be re-read against a corrected
#: mapping without measuring anything again.
NAMES = {slug: catalogue.load(slug).declare()["names"] for slug in IMPLEMENTATIONS}


def tidy(slug: str) -> pd.DataFrame:
    """One row per (implementation, rendering, quantity), with the canonical name beside it."""
    frame = MEASURED[slug]
    columns = [column for column in frame.columns if column.startswith("said_")]
    long = frame.melt(
        id_vars=["implementation", "key", "shape", "rotation", "outcome", "seconds"],
        value_vars=columns,
        var_name="said",
        value_name="value",
    )
    long["own"] = long["said"].str[len("said_") :]
    long["canonical"] = long["own"].map(NAMES[slug])
    return long.drop(columns=["said"])


SAID = pd.concat([tidy(slug) for slug in IMPLEMENTATIONS], ignore_index=True)

#: What the geometry requires, keyed the way the store keys a rendering — which is now also the way
#: the evidence keys a row, so the join below needs no translation.
TRUTH = (
    pd.read_csv(STORE / "ground_truth.csv")
    .melt(id_vars=["key"], var_name="canonical", value_name="truth")
    .dropna(subset=["truth"])
)

SHAPES = sorted(SAID["shape"].unique())
print(
    f"{len(SAID):,} measurements — {len(IMPLEMENTATIONS)} implementations "
    f"× {len(SHAPES)} shapes × {SAID['rotation'].nunique()} angles; "
    f"{len(TRUTH):,} theoretical values to compare against"
)

5,688 measurements — 6 implementations × 9 shapes × 4 angles; 1,368 theoretical values to compare against


In [3]:
def spread(values: pd.Series) -> float:
    """How far a quantity moved when the same shape was turned, relative to its own size.

    `EPS` twice over: a width under the noise floor is agreement, and the denominator never falls
    below the floor either — so a quantity hovering around nought cannot report a large percentage
    for having moved an amount nobody could measure.
    """
    values = values.dropna()
    if len(values) < 2:
        return np.nan
    width = values.max() - values.min()
    if width < EPS:
        return 0.0
    return width / max(abs(values.mean()), EPS)


def error(value: float, truth: float) -> float:
    """How far one measurement is from what the geometry requires, as a fraction of the truth.

    The same two uses of `EPS`. A disagreement smaller than the noise floor is not a disagreement,
    and a percentage is never taken against a truth smaller than the floor — which is what used to
    produce an infinity wherever the geometry required exactly nought.
    """
    gap = abs(value - truth)
    if gap < EPS:
        return 0.0
    return gap / max(abs(truth), EPS)


def as_percent(table: pd.DataFrame) -> pd.DataFrame:
    """A table of fractions, shown as percentages a reader can scan."""
    return (table * 100).round(1)


def over(table: pd.DataFrame, tolerance: float) -> pd.DataFrame:
    """Only the rows where at least one implementation is outside the tolerance.

    Filtering is deliberate: a table of two hundred rows that are all zero hides the five that
    are not.
    """
    return table[(table > tolerance).any(axis=1)]


def columns_in_order(table: pd.DataFrame) -> pd.DataFrame:
    return table.reindex(columns=[c for c in IMPLEMENTATIONS if c in table.columns])


def heading(text: str) -> None:
    display(Markdown(text))

## 1. General statistics — how much there is, and how much of it can be compared

Three different things stand between a quantity a program computes and a number this benchmark can
judge, and keeping them apart is the first thing an analysis owes a reader:

- **The catalogue may have no name for it.** A perimeter, a singularity length, the standard
  deviation of a branching angle — these are measured and stored under the implementation's own
  name, because a canonical name exists to make two numbers comparable and there is no second
  number to compare against yet.
- **No shape may settle it.** A quantity can have a catalogued name and still have no theoretical
  value here, if no drawn shape pins one down. It is reported and left unchecked.
- **The program may simply not have answered**, on any shape.

An implementation returning forty columns of which eight can be compared is in a very different
position from one returning eight that all can, and a single count of "columns" hides it.

In [4]:
def statistics(slug: str) -> dict[str, object]:
    frame = MEASURED[slug]
    said = SAID[SAID["implementation"] == slug]
    with_value = set(said.loc[said["value"].notna(), "own"])
    canonical = {own for own, name in NAMES[slug].items() if name}
    return {
        "biomarkers produced": len(NAMES[slug]),
        "…that returned a value": len(with_value),
        "matching a canonical name": len(canonical),
        "…of those, returning a value": len(with_value & canonical),
        "unnamed, but returning a value": len(with_value - canonical),
        "seconds per image": round(frame["seconds"].astype(float).mean(), 1),
    }


pd.DataFrame({slug: statistics(slug) for slug in IMPLEMENTATIONS})

,pvbm,ocular,automorph,automorphalyzer,automorphclass,vascx
biomarkers produced,30.0,18.0,18.0,54.0,18.0,20.0
…that returned a value,30.0,18.0,18.0,40.0,18.0,20.0
matching a canonical name,20.0,8.0,15.0,17.0,15.0,8.0
"…of those, returning a value",20.0,8.0,15.0,17.0,15.0,8.0
"unnamed, but returning a value",10.0,10.0,3.0,23.0,3.0,12.0
seconds per image,19.2,2.2,0.9,3.9,0.5,2.9


### 1.1 What each implementation computes that the catalogue cannot name

Naming these is the point. Each one is either a gap in the catalogue or a mapping nobody has made
yet, and the list below is what turns "eleven unnamed columns" into work somebody can actually do.
A quantity in this list is measured and stored; it is simply absent from every comparison after
this section, because there is nothing to compare it with.

In [5]:
for number, slug in enumerate(IMPLEMENTATIONS, start=1):
    said = SAID[SAID["implementation"] == slug]
    with_value = set(said.loc[said["value"].notna(), "own"])
    unnamed = sorted(own for own in with_value if not NAMES[slug].get(own))
    heading(f"#### 1.{number} {slug} — {len(unnamed)} unnamed")
    if unnamed:
        display(pd.DataFrame({"its own name for it": unnamed}))
    else:
        print("every quantity it returns has a canonical name")

#### 1.1 pvbm — 10 unnamed

,its own name for it
0,endpoints_artery
1,endpoints_vein
2,median_branching_angle_artery
3,median_branching_angle_vein
4,singularity_length_artery
5,singularity_length_vein
6,start_points_artery
7,start_points_vein
8,tortuosity_index_artery
9,tortuosity_index_vein


#### 1.2 ocular — 10 unnamed

,its own name for it
0,endpoints_artery
1,endpoints_vein
2,length_weighted_tortuosity_artery
3,length_weighted_tortuosity_vein
4,median_branching_angle_artery
5,median_branching_angle_vein
6,pooled_tortuosity_artery
7,pooled_tortuosity_vein
8,start_points_artery
9,start_points_vein


#### 1.3 automorph — 3 unnamed

...

,its own name for it
0,CRAE_Knudtson@C_artery
1,CRVE_Knudtson@C_vein
2,average_local_calibre@B_artery
3,average_local_calibre@B_binary
4,average_local_calibre@B_vein
5,average_local_calibre@C_artery
6,average_local_calibre@C_binary
7,average_local_calibre@C_vein
8,average_local_calibre@whole_artery
9,average_local_calibre@whole_binary


#### 1.5 automorphclass — 3 unnamed

,its own name for it
0,squared_curvature_tortuosity_artery
1,squared_curvature_tortuosity_vein
2,squared_curvature_tortuosity_vessels


#### 1.6 vascx — 12 unnamed

,its own name for it
0,disc_fovea_distance_center_retina
1,disc_fovea_distance_retina
2,lw_tort_dist_max_segment_len_0p15_crcl_multipl...
3,lw_tort_dist_max_segment_len_0p15_crcl_multipl...
4,lw_tort_dist_max_segment_len_0p25_crcl_multipl...
5,lw_tort_dist_max_segment_len_0p25_crcl_multipl...
6,mean_sparsity_crcl_multiplier_1p16666666667_fu...
7,mean_sparsity_vessels
8,median_temporal_angle_arteries
9,median_temporal_angle_veins


## 2. Hard failures — what raised, and on what

A failure is **not** a bad measurement, and the two are never merged. An implementation that raises
has told you it could not answer; one that returns a confident wrong number has not. This section
is the first kind only.

The table is complete rather than summarised, because it exists to be debugged from later: a reader
must be able to reproduce one row of it from the row alone.

In [6]:
troubles = []
for slug in IMPLEMENTATIONS:
    frame = MEASURED[slug]
    noted = frame[frame["note"].notna() & (frame["note"].astype(str).str.strip() != "")]
    for _, row in noted.iterrows():
        for complaint in str(row["note"]).split("; "):
            what, _, why = complaint.partition(": ")
            troubles.append(
                {
                    "implementation": slug,
                    "shape": row["shape"],
                    "rotation": row["rotation"],
                    "what it could not answer": what,
                    "what it raised": why,
                }
            )
    for _, row in frame[frame["outcome"] == "failed"].iterrows():
        troubles.append(
            {
                "implementation": slug,
                "shape": row["shape"],
                "rotation": row["rotation"],
                "what it could not answer": "everything",
                "what it raised": str(row["note"]) or "nothing came back",
            }
        )

TROUBLES = pd.DataFrame(troubles)
print(f"{len(TROUBLES)} exceptions, over {len(SAID['key'].unique())} renderings")
TROUBLES

0 exceptions, over 36 renderings


""


## 3. Section 3 — sensitivity to rotation

The geometry is identical at every angle, so **any spread at all is the implementation or the pixel
grid beneath it**. Nothing in this section touches the ground truth: it is a check a program can be
put through without anybody knowing the right answer, which is what makes it worth doing on the
quantities no shape settles.

Each cell is the **worst spread that quantity showed on any shape** — the width across the four
angles, as a percentage of its mean on that shape, pooled over every shape by taking the largest.
Pooling this way is deliberate: a quantity that turns with the image on one shape turns with the
image, and averaging that away across eight well-behaved shapes would be a way of not noticing.

Rows are filtered to those where at least one implementation exceeds **10%**
(`ROTATION_TOLERANCE`).

In [7]:
named = SAID[SAID["canonical"].notna()]
by_shape = named.groupby(["implementation", "canonical", "shape"])["value"].apply(spread)
ROTATION = columns_in_order(
    by_shape.groupby(["implementation", "canonical"]).max().unstack("implementation")
)
unsteady = over(ROTATION, ROTATION_TOLERANCE)
print(
    f"{len(unsteady)} of {len(ROTATION)} canonical biomarkers move by more than "
    f"{ROTATION_TOLERANCE:.0%} on at least one implementation, worst shape shown, in %"
)
as_percent(unsteady)

17 of 35 canonical biomarkers move by more than 10% on at least one implementation, worst shape shown, in %


implementation,pvbm,ocular,automorph,automorphalyzer,automorphclass,vascx
canonical,,,,,,
central-retinal-equivalents/knudtson/vein,4.3,NaN,NaN,10.6,NaN,0.0
fractal-dimension/box-counting/artery,NaN,NaN,20.1,80.7,80.8,NaN
fractal-dimension/box-counting/vein,NaN,NaN,25.8,8.3,8.2,NaN
fractal-dimension/box-counting/vessels,NaN,NaN,16.4,12.0,11.7,NaN
junction-counts/junctions/artery,240.0,240.0,NaN,NaN,NaN,NaN
junction-counts/junctions/vein,66.7,66.7,NaN,NaN,NaN,NaN
tortuosity/grisan-density/artery,NaN,NaN,240.0,266.5,103.6,NaN
tortuosity/grisan-density/vein,NaN,NaN,400.0,301.7,0.0,NaN
tortuosity/grisan-density/vessels,NaN,NaN,400.0,283.0,101.7,NaN


### 3.1 The same question for the quantities nothing can compare

A quantity the catalogue cannot name is as capable of turning with the image as one it can, and
nothing else in this notebook would notice. This is the only check those columns get.

In [8]:
unnamed_rows = SAID[SAID["canonical"].isna() & SAID["value"].notna()]
for number, slug in enumerate(IMPLEMENTATIONS, start=1):
    mine = unnamed_rows[unnamed_rows["implementation"] == slug]
    heading(f"#### 3.{number} {slug}")
    if mine.empty:
        print("nothing unnamed to check")
        continue
    table = mine.groupby(["own", "shape"])["value"].apply(spread).groupby("own").max()
    table = table[table > ROTATION_TOLERANCE].sort_values(ascending=False)
    if table.empty:
        print(f"every unnamed quantity holds steady to within {ROTATION_TOLERANCE:.0%}")
    else:
        display((table * 100).round(1).to_frame("worst spread across angles, %"))

#### 3.1 pvbm

,"worst spread across angles, %"
own,
singularity_length_vein,27.0
singularity_length_artery,26.4
tortuosity_index_artery,14.5
endpoints_vein,12.9
median_branching_angle_artery,10.3


#### 3.2 ocular

,"worst spread across angles, %"
own,
pooled_tortuosity_artery,14.5
endpoints_vein,12.9
length_weighted_tortuosity_artery,11.8
median_branching_angle_artery,10.3


#### 3.3 automorph

...

,"worst spread across angles, %"
own,
tortuosity_density@B_artery,400.0
tortuosity_density@B_vein,400.0
tortuosity_density@C_artery,400.0
tortuosity_density@C_vein,400.0
tortuosity_density@C_binary,330.8
tortuosity_density@B_binary,318.8
average_local_calibre@whole_vein,26.7
CRVE_Knudtson@C_vein,14.1
average_local_calibre@whole_artery,14.1


#### 3.5 automorphclass

,"worst spread across angles, %"
own,
squared_curvature_tortuosity_vein,203.1
squared_curvature_tortuosity_vessels,192.8
squared_curvature_tortuosity_artery,189.2


#### 3.6 vascx

,"worst spread across angles, %"
own,
median_temporal_angle_arteries,39.8


### 3.2 The single worst case each implementation produced

The tables above give a spread as a percentage, which says *how much* a quantity moved and nothing
about what it moved **between**. A spread of 240% could be 1 against 3.4, or 0.001 against 0.0034,
and those are not the same finding: the first is a count that cannot make up its mind and the
second is noise in a number nobody would quote to three figures.

So this is the worst case each implementation produced anywhere — the quantity, the shape, and the
two angles that bracket it, **with the values themselves**. Every quantity is eligible, canonical
or not, because a program's worst behaviour does not check whether the catalogue has a name for it.

Read it as a place to start debugging rather than as a score. One bad shape is one bad shape.

In [9]:
def worst_rotation_case(slug: str) -> dict[str, object] | None:
    """The (quantity, shape) whose value moved most across the four angles."""
    mine = SAID[(SAID["implementation"] == slug) & SAID["value"].notna()]
    best = None
    for (own, shape), group in mine.groupby(["own", "shape"]):
        values = group.dropna(subset=["value"])
        if len(values) < 2:
            continue
        width = values["value"].max() - values["value"].min()
        mean = values["value"].mean()
        if width < EPS:
            continue          # under the noise floor: not a disagreement worth ranking
        spread = width / max(abs(mean), EPS)
        if best is None or spread > best["spread"]:
            low = values.loc[values["value"].idxmin()]
            high = values.loc[values["value"].idxmax()]
            best = {
                "quantity": own,
                "canonical": group["canonical"].iloc[0] or "—",
                "shape": shape,
                "spread": spread,
                "at min": f"{low['rotation']:.0f}°",
                "min": low["value"],
                "at max": f"{high['rotation']:.0f}°",
                "max": high["value"],
            }
    return best


rows = []
for slug in IMPLEMENTATIONS:
    found = worst_rotation_case(slug)
    if found is None:
        continue
    rows.append({"implementation": slug, **found})

WORST = pd.DataFrame(rows).set_index("implementation")
WORST["spread"] = (100 * WORST["spread"]).round(0).astype(int).astype(str) + "%"
WORST[["quantity", "shape", "spread", "at min", "min", "at max", "max", "canonical"]]

,quantity,shape,spread,at min,min,at max,max,canonical
implementation,,,,,,,,
pvbm,intersections_artery,koch,240%,30°,0.000000,0°,3.000000,junction-counts/junctions/artery
ocular,intersections_artery,koch,240%,30°,0.000000,0°,3.000000,junction-counts/junctions/artery
automorph,squared_curvature_tortuosity_artery,sinusoid,400%,0°,0.000000,30°,0.089443,—
automorphalyzer,tortuosity_density@B_artery,straight,400%,0°,0.000000,60°,0.992366,—
automorphclass,squared_curvature_tortuosity_vein,straight,203%,0°,0.000000,60°,5.690000,—
vascx,lw_tort_curv_crcl_multiplier_1p16666666667_ful...,spokes-macula-centred,280%,30°,0.000941,0°,0.066056,tortuosity/spline-mean-curvature/artery


## 4. Section 4 — agreement with ground truth

Here the number on the other side of the comparison is not another program's: it is what the
geometry **requires**, worked out from the equations the shape was drawn from. A disagreement is
therefore a fault rather than a difference of opinion.

**The main table holds the worst disagreement over every shape and every angle** — one number per
implementation per biomarker, the worst case it produced anywhere, as a percentage of the truth.
Rows are filtered to those with at least one fault, where a fault is more than **25%**
(`GROUND_TRUTH_TOLERANCE`).

Only canonical names appear, because only they have a ground truth to disagree with. A quantity
computed under a name the catalogue does not know is measured, stored, and absent from here — which
is a gap rather than a pass, and section 1's subsections say which quantities those are.

Where the geometry requires exactly nought — a straight vessel has no inflections — there is no
percentage of nought to take, so the denominator falls back to the noise floor `EPS`. A program
that finds structure in a shape that has none therefore reports a large but finite error rather
than an infinity, and how large depends on the floor rather than on the truth. Section 4.2 shows
the values themselves, which is the only way to read those rows.

In [10]:
AGAINST = named.merge(TRUTH, on=["key", "canonical"], how="inner").dropna(subset=["value"])
AGAINST["error"] = [error(v, t) for v, t in zip(AGAINST["value"], AGAINST["truth"])]


def worst_over(frame: pd.DataFrame) -> pd.DataFrame:
    """The worst disagreement each implementation produced, per canonical name."""
    return columns_in_order(
        frame.groupby(["canonical", "implementation"])["error"].max().unstack("implementation")
    )


EVERYWHERE = worst_over(AGAINST)
faults = over(EVERYWHERE, GROUND_TRUTH_TOLERANCE)
print(
    f"{len(AGAINST):,} measurements have a theoretical value to be judged against, over "
    f"{AGAINST['canonical'].nunique()} canonical biomarkers; {len(faults)} of them are out by more "
    f"than {GROUND_TRUTH_TOLERANCE:.0%} somewhere. Worst over all shapes × angles, in %:"
)
as_percent(faults)

1,296 measurements have a theoretical value to be judged against, over 28 canonical biomarkers; 13 of them are out by more than 25% somewhere. Worst over all shapes × angles, in %:


implementation,pvbm,ocular,automorph,automorphalyzer,automorphclass,vascx
canonical,,,,,,
avr/hubbard/both,32.2,NaN,NaN,NaN,NaN,NaN
central-retinal-equivalents/hubbard/artery,81.4,NaN,NaN,NaN,NaN,NaN
central-retinal-equivalents/hubbard/vein,72.5,NaN,NaN,NaN,NaN,NaN
junction-counts/junctions/artery,42.9,42.9,NaN,NaN,NaN,NaN
junction-counts/junctions/vein,85.7,85.7,NaN,NaN,NaN,NaN
tortuosity/grisan-density/artery,NaN,NaN,8285.7,9977.3,103.8,NaN
tortuosity/grisan-density/vein,NaN,NaN,9285.7,9940.3,0.0,NaN
tortuosity/hart-tau1/artery,16.5,16.5,105.3,13.7,8.5,43.5
tortuosity/hart-tau1/vein,31.2,31.2,542.2,13.4,7.4,43.4


### 4.1 onwards — the same table, one shape at a time

The main table says **whether** a biomarker can be trusted. It cannot say **where** it stopped
being trustworthy, and that is the difference between a number and something somebody can go and
investigate: a biomarker wrong on one shape and right on eight is a different finding from one
wrong on all nine, and the worst-case column above cannot tell them apart.

Each table below is the worst disagreement over the four angles of that shape alone.

In [11]:
for number, shape in enumerate(SHAPES, start=1):
    here = AGAINST[AGAINST["shape"] == shape]
    table = over(worst_over(here), GROUND_TRUTH_TOLERANCE)
    heading(f"#### 4.{number} {shape} — {len(table)} biomarkers out by more than {GROUND_TRUTH_TOLERANCE:.0%}")
    if table.empty:
        print("every comparable biomarker lands within tolerance on this shape")
    else:
        display(as_percent(table))

#### 4.1 arc — 4 biomarkers out by more than 25%

implementation,pvbm,ocular,automorph,automorphalyzer,automorphclass
canonical,,,,,
tortuosity/grisan-density/artery,NaN,NaN,8285.7,9963.8,0.0
tortuosity/grisan-density/vein,NaN,NaN,9285.7,9104.4,0.0
tortuosity/hart-tau1/artery,5.3,5.3,105.3,5.3,5.7
tortuosity/hart-tau1/vein,5.1,5.1,542.2,3.3,5.5


#### 4.2 bifurcation — 1 biomarkers out by more than 25%

implementation,pvbm,ocular,automorph,automorphalyzer,automorphclass
canonical,,,,,
vessel-calibre/mean-width/vein,NaN,NaN,28.1,12.5,12.0


#### 4.3 deep-bifurcation — 4 biomarkers out by more than 25%

...

implementation,pvbm,ocular,automorph,automorphalyzer,automorphclass,vascx
canonical,,,,,,
avr/hubbard/both,32.2,NaN,NaN,NaN,NaN,NaN
central-retinal-equivalents/hubbard/artery,81.4,NaN,NaN,NaN,NaN,NaN
central-retinal-equivalents/hubbard/vein,72.5,NaN,NaN,NaN,NaN,NaN


#### 4.8 spokes-macula-centred — 5 biomarkers out by more than 25%

implementation,pvbm,ocular,automorph,automorphalyzer,automorphclass,vascx
canonical,,,,,,
avr/hubbard/both,32.2,NaN,NaN,NaN,NaN,NaN
central-retinal-equivalents/hubbard/artery,81.4,NaN,NaN,NaN,NaN,NaN
central-retinal-equivalents/hubbard/vein,72.5,NaN,NaN,NaN,NaN,NaN
vessel-calibre/mean-width/artery,NaN,NaN,31.6,10.0,13.4,0.7
vessel-calibre/mean-width/vein,NaN,NaN,56.5,10.4,15.4,0.7


#### 4.9 straight — 5 biomarkers out by more than 25%

implementation,pvbm,ocular,automorph,automorphalyzer,automorphclass
canonical,,,,,
tortuosity/grisan-density/artery,NaN,NaN,0.0,9977.3,103.6
tortuosity/grisan-density/vein,NaN,NaN,0.0,6103.5,0.0
tortuosity/hart-tau1/artery,7.3,7.3,100.0,7.5,7.5
tortuosity/hart-tau1/vein,7.3,7.3,100.0,7.1,7.3
vessel-calibre/mean-width/vein,NaN,NaN,41.0,13.4,18.8


### 4.2 The single worst disagreement each implementation produced

The same objection as section 3.2, and the same answer. A table of percentages says how far a
number is from the geometry and not what either number **was**, so a 100% error could be 2 against
4 or 0 against anything at all — and "it returned nothing where the geometry requires something" is
a different fault from "it returned twice too much".

So this is the worst single disagreement each implementation produced anywhere: the quantity under
**its own name**, the catalogued name it answers to, the shape and angle it happened on, and both
values side by side.

Only canonical names can appear, because only they have a ground truth to disagree with. `inf`
means the geometry requires exactly nought and the implementation returned something else, which is
why the two value columns matter more than the percentage.

In [12]:
def worst_truth_case(slug: str) -> dict[str, object] | None:
    """The single measurement furthest from what the geometry requires."""
    mine = AGAINST[AGAINST["implementation"] == slug]
    if mine.empty:
        return None
    row = mine.loc[mine["error"].idxmax()]
    return {
        "quantity": row["own"],
        "canonical": row["canonical"],
        "shape": row["shape"],
        "rotation": f"{row['rotation']:.0f}°",
        "ground truth": row["truth"],
        "it returned": row["value"],
        "error": f"{100 * row['error']:.0f}%",
    }


rows = [
    {"implementation": slug, **found}
    for slug in IMPLEMENTATIONS
    if (found := worst_truth_case(slug)) is not None
]
WORST_TRUTH = pd.DataFrame(rows).set_index("implementation")
WORST_TRUTH[["quantity", "canonical", "shape", "rotation", "ground truth", "it returned", "error"]]

,quantity,canonical,shape,rotation,ground truth,it returned,error
implementation,,,,,,,
pvbm,length_artery,vessel-area-and-length/skeleton-length/artery,disjoint,0°,2785.280000,0.000000,100%
ocular,length_artery,vessel-area-and-length/skeleton-length/artery,disjoint,0°,2785.280000,0.000000,100%
automorph,tortuosity_density_vein,tortuosity/grisan-density/vein,arc,90°,0.000000,0.928571,9286%
automorphalyzer,tortuosity_density@whole_artery,tortuosity/grisan-density/artery,straight,60°,0.000000,0.997732,9977%
automorphclass,tortuosity_density_artery,tortuosity/grisan-density/artery,sinusoid,30°,0.001105,0.011488,104%
vascx,lw_tort_dist_max_segment_len_0p2_crcl_multipli...,tortuosity/hart-tau1/artery,koch,0°,1.777778,1.005314,43%


## 5. Summary

Two counts, each written as a fraction so that the denominator is visible. **The denominators are
different on purpose**: rotation can be checked on anything that returned a number at all, and
agreement only on what the catalogue can name *and* a shape can settle. Reporting both against one
denominator would make one of them a lie.

In [13]:
#: Rotation spread for **every** quantity, named or not — which is the denominator the summary
#: needs, because a self-consistency check does not require a canonical name.
returned = SAID[SAID["value"].notna()]
EVERY_ROTATION = (
    returned.groupby(["implementation", "own", "shape"])["value"]
    .apply(spread)
    .groupby(["implementation", "own"])
    .max()
)

summary = []
for slug in IMPLEMENTATIONS:
    steady = EVERY_ROTATION.loc[slug].dropna()
    judged = EVERYWHERE[slug].dropna()
    mine = AGAINST[AGAINST["implementation"] == slug]
    summary.append(
        {
            "implementation": slug,
            "rotation problems": f"{int((steady > ROTATION_TOLERANCE).sum())} / {len(steady)}",
            "disagrees with geometry": f"{int((judged > GROUND_TRUTH_TOLERANCE).sum())} / {len(judged)}",
            "exceptions": int((TROUBLES["implementation"] == slug).sum()) if len(TROUBLES) else 0,
            # Not a score, and not comparable across columns without reading what each column
            # *contains*: an implementation measuring six quantities well is not better than one
            # measuring twenty-two adequately, it is answering a smaller question.
            "comparable measurements": int(mine.shape[0]),
            "median disagreement": f"{np.median(mine['error']) * 100:.1f}%",
            "within 25% of the geometry": f"{(mine['error'] <= GROUND_TRUTH_TOLERANCE).mean():.0%}",
            "seconds per image": round(MEASURED[slug]["seconds"].astype(float).mean(), 1),
        }
    )

pd.DataFrame(summary).set_index("implementation").T

implementation,pvbm,ocular,automorph,automorphalyzer,automorphclass,vascx
rotation problems,9 / 30,8 / 18,15 / 18,19 / 40,10 / 18,4 / 20
disagrees with geometry,8 / 20,5 / 8,6 / 10,2 / 12,2 / 10,2 / 6
exceptions,0,0,0,0,0,0
comparable measurements,312,240,216,232,216,80
median disagreement,3.8%,2.2%,3.9%,6.5%,5.3%,0.7%
within 25% of the geometry,87%,93%,79%,91%,98%,90%
seconds per image,19.2,2.2,0.9,3.9,0.5,2.9


### 5.1 What this does not say

- **It does not rank the implementations.** No column is scored and no order is published. What can
  legitimately be argued from this evidence is *which implementation to reach for at a named
  question* — calibre, or tortuosity, or surviving a rotation — with the caveats that would change
  the answer. That argument belongs on the results page, where a reader can disagree with it. A
  column sorted by score would not be an argument at all.
- **It does not say a quantity with no ground truth is wrong**, or right. It is unchecked, which is
  a third thing.
- **It does not treat agreement between two implementations as evidence that either is correct.**
  Two of these groups share a lineage: where siblings agree, that is evidence about their common
  ancestor and nothing more.
- **It does not say anything about photographs.** Every shape here is clean, complete and drawn to
  a known scale. A program that measures a drawn arc correctly has cleared the lowest bar there is,
  not the one that matters clinically.